# Colorizar una foto familiar en blanco y negro

Este cuaderno construye el MVP paso a paso: una persona de la familia carga una foto, un modelo de difusión propone colores, CLIP compara la imagen antes y después y Gradio muestra ambas versiones.

> **Importante:** una foto en blanco y negro no contiene sus colores originales. El modelo genera una propuesta verosímil, no recupera ni verifica la verdad histórica. Conservá el original.

El trabajo usa un ControlNet de Hugging Face y un modelo base Stable Diffusion 2.1. La ficha de ControlNet solo declara licencia OpenRAIL y casi no documenta sus datos o resultados; lo probaremos como experimento, no como una solución validada. Sus pesos ocupan alrededor de 1,46 GB, además de los varios GB del modelo base.

## Antes de empezar en Google Colab

1. Abrí este archivo en Colab desde el botón **Abrir en Colab** del README.
2. En **Entorno de ejecución → Cambiar tipo de entorno de ejecución**, elegí **GPU** (por ejemplo, T4 si está disponible).
3. Ejecutá las celdas en orden, desde arriba. La primera instala las dependencias; luego se descargan los modelos, varios GB en total.
4. Al llegar a la interfaz, cargá una imagen propia o con permiso. No uses fotos familiares privadas sin autorización: Colab es un entorno remoto.

La sesión y `/content` son temporales: si Colab se desconecta, se pierden la caché y los archivos de esa sesión. La copia local original no cambia. Los modelos usados son públicos y no necesitan token de Hugging Face.

Recorrido: instalación → rutas temporales y dispositivo → modelos → preparación de imagen → colorización → comparación y CLIP → interfaz Gradio dentro de Colab.

## 0. Instalar dependencias en Colab

Colab ya proporciona PyTorch compilado para el runtime; no lo reinstalamos para evitar reemplazar su versión con CUDA. Esta celda instala el resto de las librerías del proyecto. Ejecutala primero y, si Colab solicita reiniciar el entorno, reinicialo y continuá desde la primera celda.

In [ ]:
%pip install -q "diffusers>=0.32,<1" "transformers>=4.49,<5" "accelerate>=1.2,<2" "safetensors>=0.5,<1" "gradio>=5.16,<6" "pillow>=11,<13" "numpy>=1.26,<3"

## 1. Preparar almacenamiento temporal

En Colab usamos `/content`, que está disponible durante esta sesión. Las cachés y resultados no se guardan en la computadora local ni persisten al desconectarse. Ejecutá esta celda antes de importar Diffusers o Transformers.

In [ ]:
import os
from pathlib import Path

ROOT = Path("/content")
CACHE_DIR = ROOT / ".model-cache"
HF_HOME = CACHE_DIR / "huggingface"
OUTPUT_DIR = ROOT / "resultados"
GRADIO_TEMP_DIR = CACHE_DIR / "gradio"
for carpeta in (HF_HOME, OUTPUT_DIR, CACHE_DIR / "torch", GRADIO_TEMP_DIR):
    carpeta.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["TORCH_HOME"] = str(CACHE_DIR / "torch")
os.environ["GRADIO_TEMP_DIR"] = str(GRADIO_TEMP_DIR)

print("Proyecto temporal:", ROOT)
print("Caché de modelos:", HF_HOME)
print("Resultados:", OUTPUT_DIR)

## 2. Comprobar el entorno GPU

Este cuaderno está preparado para una GPU de Colab. Si no se detecta CUDA, detenemos la ejecución en vez de iniciar por accidente una inferencia que en CPU puede tardar horas. En Colab, elegí **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU** y volvé a ejecutar esta celda.

In [ ]:
import torch
from PIL import Image, ImageOps
from IPython.display import display

if not torch.cuda.is_available():
    raise RuntimeError("No se detecto GPU. En Colab, activa un acelerador GPU y vuelve a ejecutar las celdas.")

DEVICE = "cuda"
DTYPE = torch.float16
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("Dispositivo:", DEVICE)
print("GPU:", GPU_NAME)
print(f"Memoria GPU: {GPU_MEMORY_GB:.1f} GB")

## 3. Modelos públicos de Hugging Face

El modelo base [`sd2-community/stable-diffusion-2-1-base`](https://huggingface.co/sd2-community/stable-diffusion-2-1-base) y el ControlNet de colorización son públicos en Hugging Face: no hace falta iniciar sesión ni ingresar un token. Ambos publican sus pesos bajo licencias OpenRAIL; revisá sus condiciones de uso. El repositorio base informa la arquitectura Stable Diffusion 2.1, compatible con el ControlNet elegido.

## 4. Cargar el modelo de difusión

ControlNet recibe la estructura de la foto en escala de grises y condiciona la generación. El modelo base propone los colores. Fijamos las revisiones de ambos repositorios para que la descarga sea reproducible. La primera descarga del modelo base ocupa varios GB.

In [ ]:
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline

CONTROLNET_ID = "neurallove/controlnet-sd21-colorization-diffusers"
CONTROLNET_REVISION = "2083978ca9345c9edac3af2779c17710b7e42258"
BASE_MODEL_ID = "sd2-community/stable-diffusion-2-1-base"
BASE_MODEL_REVISION = "4e63672c03103b6c636b8fb4119ba982469b2955"

controlnet = ControlNetModel.from_pretrained(
    CONTROLNET_ID,
    revision=CONTROLNET_REVISION,
    use_safetensors=False,
    torch_dtype=DTYPE,
)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    BASE_MODEL_ID,
    revision=BASE_MODEL_REVISION,
    controlnet=controlnet,
    torch_dtype=DTYPE,
)
pipe = pipe.to(DEVICE)
pipe.enable_attention_slicing()
pipe.set_progress_bar_config(disable=False)

PIPELINE_LISTO = True
print("Modelo de difusión cargado en", DEVICE)

## 5. Preparar una imagen sin deformarla

Convertimos la entrada a blanco y negro para que el experimento sea consistente. Reducimos el lado mayor a 512 píxeles y redondeamos dimensiones a múltiplos de 8, que son adecuados para la arquitectura de Stable Diffusion. El resultado se ampliará a las dimensiones originales para poder compararlo.

In [ ]:
def preparar_imagen(imagen, lado_maximo=512):
    """Devuelve la foto gris y su versión reducida para el modelo."""
    original = ImageOps.grayscale(imagen).convert("RGB")
    ancho, alto = original.size
    escala = min(1.0, lado_maximo / max(ancho, alto))
    nuevo_ancho = max(64, int(ancho * escala) // 8 * 8)
    nuevo_alto = max(64, int(alto * escala) // 8 * 8)
    reducida = original.resize((nuevo_ancho, nuevo_alto), Image.Resampling.LANCZOS)
    return original, reducida

print("Función de preparación lista. La interfaz permitirá cargar una imagen cuando esté completa.")

## 6. Generar la propuesta de color

La semilla hace que una repetición con los mismos parámetros produzca un resultado más comparable. El prompt orienta el estilo; no aporta información verdadera sobre los colores de la escena. Bajá los pasos para una prueba rápida, sabiendo que eso también puede cambiar el resultado.

In [ ]:
PROMPT = "colorized vintage black and white family photograph, natural realistic colors, preserve the original composition and details"
NEGATIVE_PROMPT = "black and white, grayscale, sepia, changed composition, distorted face, artifacts, text"

def colorizar(imagen, semilla=42, pasos=15):
    if "pipe" not in globals():
        raise RuntimeError("Primero ejecutá la celda que carga el modelo de difusión.")
    original, control = preparar_imagen(imagen)
    generador = torch.Generator(device=DEVICE).manual_seed(int(semilla))
    resultado = pipe(
        prompt=PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        image=control,
        width=control.width,
        height=control.height,
        num_inference_steps=int(pasos),
        guidance_scale=7.5,
        controlnet_conditioning_scale=1.0,
        generator=generador,
    ).images[0]
    resultado = resultado.resize(original.size, Image.Resampling.LANCZOS)
    return original, resultado

print("Función de colorización lista; todavía no se descargó ni ejecutó ninguna foto.")

## 7. Preparar el análisis visual con CLIP

CLIP compara la imagen con dos descripciones de texto. Sus puntajes expresan afinidad relativa con esas etiquetas: **no** miden calidad, naturalidad, identidad ni autenticidad de los colores. Cargamos este segundo modelo desde Hugging Face; también descarga sus propios pesos.

In [ ]:
from transformers import CLIPModel, CLIPProcessor

CLIP_ID = "openai/clip-vit-base-patch32"
clip_processor = CLIPProcessor.from_pretrained(CLIP_ID)
clip_model = CLIPModel.from_pretrained(CLIP_ID, torch_dtype=DTYPE).to(DEVICE).eval()
CLIP_LISTO = True
print("CLIP cargado en", DEVICE)

In [ ]:
ETIQUETAS_CLIP = ["a black and white photograph", "a color photograph"]

@torch.inference_mode()
def analizar_con_clip(imagenes):
    entradas = clip_processor(
        text=ETIQUETAS_CLIP,
        images=imagenes,
        return_tensors="pt",
        padding=True,
    )
    entradas = {nombre: tensor.to(DEVICE) for nombre, tensor in entradas.items()}
    puntajes = clip_model(**entradas).logits_per_image.softmax(dim=1).cpu().tolist()
    return puntajes

print("Analizador CLIP listo.")

## 8. Preparar la comparación

Esta función reúne procesamiento y análisis para que la use tanto el cuaderno como Gradio. En la tabla veremos qué etiqueta se parece más a cada imagen; el puntaje no sirve como verificación del color real.

In [ ]:
import time

PRUEBA_OK = False

def procesar_foto(imagen, semilla=42, pasos=15):
    global PRUEBA_OK
    if imagen is None:
        raise ValueError("Carga una imagen para comenzar.")
    inicio = time.perf_counter()
    original, colorizada = colorizar(imagen, semilla=semilla, pasos=pasos)
    puntajes = analizar_con_clip([original, colorizada])
    tiempo_total = time.perf_counter() - inicio
    nombres = ["Original", "Colorizada"]
    reporte = "| Imagen | Blanco y negro | A color |\n|---|---:|---:|\n"
    for nombre, fila in zip(nombres, puntajes):
        reporte += f"| {nombre} | {fila[0]:.1%} | {fila[1]:.1%} |\n"
    reporte += f"\n**Tiempo total (generacion + CLIP): {tiempo_total:.1f} s. GPU: {GPU_NAME} ({GPU_MEMORY_GB:.1f} GB).**"
    reporte += "\n\n*CLIP estima afinidad con dos etiquetas; no comprueba los colores originales.*"
    PRUEBA_OK = True
    return original, colorizada, reporte

print("Flujo integrado listo. Al procesar una foto, se informaran GPU, tiempo y puntajes CLIP.")

## 9. Abrir la interfaz en Colab

Gradio se muestra dentro del cuaderno a través del proxy de Colab; no hace falta crear un enlace público. La imagen se procesa en un runtime remoto de Google. Cargá solo imágenes propias o con permiso y no incluyas fotografías familiares privadas sin autorización.

In [ ]:
import gradio as gr

with gr.Blocks(title="Colorizador de fotos familiares") as demo:
    gr.Markdown(
        "# Colorizador de fotos familiares\n"
        "**Los colores son una propuesta de IA, no los colores históricos comprobados. Conservá tu original.**"
    )
    with gr.Row():
        entrada = gr.Image(label="Foto en blanco y negro", type="pil")
        original_salida = gr.Image(label="Original en escala de grises", type="pil")
        color_salida = gr.Image(label="Propuesta colorizada", type="pil")
    pasos = gr.Slider(5, 25, value=15, step=1, label="Pasos de difusión (más pasos pueden tardar más)")
    semilla = gr.Number(value=42, precision=0, label="Semilla")
    boton = gr.Button("Colorizar y analizar")
    analisis = gr.Markdown()
    boton.click(
        fn=procesar_foto,
        inputs=[entrada, semilla, pasos],
        outputs=[original_salida, color_salida, analisis],
    )

demo.launch(share=False, inline=True, debug=True)

## 10. Qué observar y explicar

1. ¿Qué formas y tonos de la foto original se conservaron?
2. ¿Qué zonas recibieron colores que no podemos comprobar?
3. ¿Cambió CLIP su afinidad hacia “a color”? Eso describe la etiqueta, no la calidad.
4. ¿Qué pasa al repetir con otra semilla o cantidad de pasos?
5. ¿Qué GPU y tiempo total reportó la interfaz? Registralos junto con los parámetros y guardá una captura para documentar la ejecución en Colab.

La foto se procesa en el runtime remoto de Colab. No se crea un enlace público de Gradio. El modelo agrega contenido plausible y puede alterar detalles; compará siempre con el original y conservá la foto fuente.